In [ ]:
# ============================================================
# CELLULE 1 — Imports et chargement du dataset brut
# ============================================================
# Objectif : charger le fichier CSV original et avoir un premier aperçu.
# On garde df sous la main, on n'y touche pas (on créera df_clean plus tard).

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Petits réglages d'affichage pour avoir des graphes lisibles
plt.rcParams['figure.figsize'] = (10, 6)
sns.set_theme(style='whitegrid', palette='muted')

# Lecture du dataset brut (10 000 clients, 32 colonnes)
df = pd.read_csv('../data/raw/customer_churn.csv')

print(f"Dimensions : {df.shape[0]} lignes × {df.shape[1]} colonnes")
print()
display(df.head())   # aperçu rapide pour repérer les colonnes


In [ ]:
# ============================================================
# CELLULE 2 — Audit qualité du dataset
# ============================================================
# Trois questions à se poser avant tout :
#   1. Quels sont les types de chaque colonne ? (numérique vs catégorielle)
#   2. Y a-t-il des valeurs manquantes à traiter ?
#   3. Les distributions ont-elles l'air cohérentes (pas d'aberrations) ?

print("=== Types de données ===")
print(df.dtypes)
print()

# Recherche des NaN — important car certains modèles ne les supportent pas
missing = df.isnull().sum()
print("=== Valeurs manquantes ===")
print("Aucune valeur manquante !" if missing.sum() == 0 else missing[missing > 0])
# Note : on s'attend à voir des NaN sur complaint_type
# (logique métier : un client sans plainte n'a pas de type de plainte)

print()
print("=== Statistiques descriptives ===")
# describe() donne min/max/quartiles → repère vite les colonnes à scaler
display(df.describe())


In [ ]:
# ============================================================
# CELLULE 3 — Distribution de la variable cible (churn)
# ============================================================
# C'est LA première chose à regarder dans un projet de classification :
# le dataset est-il équilibré ?
# Spoiler : non. ~10% de churners — il faudra en tenir compte
# (class_weight='balanced', scale_pos_weight, etc.).

churn_counts = df['churn'].value_counts()
churn_pct    = df['churn'].value_counts(normalize=True) * 100

print(f"Non-churn (0) : {churn_counts[0]} clients  ({churn_pct[0]:.1f}%)")
print(f"Churn     (1) : {churn_counts[1]} clients  ({churn_pct[1]:.1f}%)")

# Double visualisation : barres (volume) + camembert (proportion)
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].bar(['Non-Churn (0)', 'Churn (1)'], churn_counts,
            color=['#2196F3', '#E53935'], edgecolor='white', linewidth=1.5)
axes[0].set_title('Nombre de clients par classe', fontweight='bold')
axes[0].set_ylabel('Nombre de clients')
# Petite annotation pour afficher les valeurs au-dessus des barres
for i, v in enumerate(churn_counts):
    axes[0].text(i, v + 30, str(v), ha='center', fontweight='bold')

axes[1].pie(churn_counts, labels=['Non-Churn', 'Churn'],
            colors=['#2196F3', '#E53935'],
            autopct='%1.1f%%', startangle=90,
            wedgeprops={'edgecolor': 'white', 'linewidth': 2})
axes[1].set_title('Proportion des classes', fontweight='bold')

plt.suptitle('Distribution de la variable cible : Churn', fontsize=14, fontweight='bold')
plt.tight_layout()
# On sauvegarde le graphe pour le réutiliser dans le rapport / dashboard
plt.savefig('../data/processed/churn_distribution.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# ============================================================
# CELLULE 4 — Distributions numériques par classe (churn vs non-churn)
# ============================================================
# Idée : pour chaque variable numérique, on superpose la distribution
# des churners (rouge) et des non-churners (bleu).
# Si les deux courbes se chevauchent → la variable n'a pas grand intérêt.
# Si elles se séparent nettement → variable prédictive intéressante.

# On exclut churn de la liste car c'est la cible, pas une feature
num_cols = [c for c in df.select_dtypes(include=['int64','float64']).columns
            if c != 'churn']
print(f"Variables numériques ({len(num_cols)}) : {num_cols}")

# Mise en page automatique : 3 colonnes, autant de lignes que nécessaire
n_cols = 3
n_rows = (len(num_cols) + n_cols - 1) // n_cols
fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, n_rows * 4))
axes = axes.flatten()

for i, col in enumerate(num_cols):
    # density=True normalise les histogrammes → comparable malgré l'écart de volume (10% / 90%)
    df[df['churn']==0][col].hist(ax=axes[i], alpha=0.6, color='#2196F3',
                                  label='Non-Churn', bins=30, density=True)
    df[df['churn']==1][col].hist(ax=axes[i], alpha=0.6, color='#E53935',
                                  label='Churn', bins=30, density=True)
    axes[i].set_title(col, fontweight='bold')
    axes[i].legend(fontsize=8)

# On masque les sous-graphes vides en bas à droite
for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Distributions numériques par classe de churn', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../data/processed/numerical_distributions.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# ============================================================
# CELLULE 5 — Taux de churn par variable catégorielle
# ============================================================
# Pour chaque variable catégorielle, on calcule le taux de churn moyen
# par modalité. Plus l'écart entre les barres est grand, plus la variable
# est discriminante. Exemple typique : contract_type "Monthly" >> "Yearly".

cat_cols = df.select_dtypes(include=['object']).columns.tolist()
print(f"Variables catégorielles ({len(cat_cols)}) : {cat_cols}")

if cat_cols:
    fig, axes = plt.subplots(1, len(cat_cols), figsize=(5 * len(cat_cols), 5))
    # Sécurité : si une seule colonne, axes n'est pas un tableau
    if len(cat_cols) == 1:
        axes = [axes]
    for i, col in enumerate(cat_cols):
        # mean() sur churn (0/1) → renvoie directement le taux de churn par modalité
        churn_rate = df.groupby(col)['churn'].mean() * 100
        churn_rate.sort_values(ascending=False).plot(
            kind='bar', ax=axes[i], color='#7B1FA2', edgecolor='white')
        axes[i].set_title(f'Taux de churn — {col}', fontweight='bold')
        axes[i].set_ylabel('Taux de churn (%)')
        # Rotation des labels pour qu'ils restent lisibles
        axes[i].tick_params(axis='x', rotation=30)
    plt.suptitle('Taux de churn par variable catégorielle', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()


In [ ]:
# ============================================================
# CELLULE 6 — Matrice de corrélation
# ============================================================
# Deux objectifs :
#   1. Repérer les variables les plus corrélées avec le churn
#      → candidates naturelles pour le modèle.
#   2. Détecter les paires de variables très corrélées entre elles
#      (multicolinéarité) → on en garde une seule pour éviter le bruit.

corr_matrix = df[num_cols + ['churn']].corr()

plt.figure(figsize=(13, 10))
# Masque triangulaire : on n'affiche que la moitié inférieure (la matrice est symétrique)
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f',
            cmap='RdBu_r', center=0, square=True,
            linewidths=0.5, cbar_kws={'shrink': 0.8})
plt.title('Matrice de corrélation', fontweight='bold', fontsize=14)
plt.tight_layout()
plt.savefig('../data/processed/correlation_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

# Classement explicite des features par corrélation avec le churn
# (positives = augmentent le risque ; négatives = protectrices)
print("=== Top corrélations avec le churn ===")
print(corr_matrix['churn'].drop('churn').sort_values(ascending=False).to_string())


In [ ]:
# ============================================================
# CELLULE 7 — Nettoyage et feature engineering
# ============================================================
# Maintenant que l'EDA est faite, on prépare un dataset propre prêt à modéliser.
# On travaille sur df_clean (copie) pour ne pas écraser le df original.

df_clean = df.copy()

# 1. Suppression des colonnes inutiles pour la modélisation
#    - customer_id : identifiant unique, aucun pouvoir prédictif
#    - city / country : trop de modalités → exploserait le nombre de colonnes
#      après one-hot, et l'apport métier est faible ici.
cols_to_drop = ['customer_id', 'city', 'country']
df_clean = df_clean.drop(columns=cols_to_drop)
print(f"Colonnes supprimées : {cols_to_drop}")

# 2. Remplissage des NaN sur complaint_type
#    Logique métier : une valeur manquante = aucune plainte enregistrée.
#    On crée donc une nouvelle modalité explicite "No_complaint" plutôt
#    que de supprimer ces lignes (~20% du dataset, ce serait dommage).
df_clean['complaint_type'] = df_clean['complaint_type'].fillna('No_complaint')
print(f"Valeurs manquantes restantes : {df_clean.isnull().sum().sum()}")

# 3. Feature engineering — 3 variables métier qui apportent du signal
#    On les construit à partir de combinaisons "intelligentes" des features brutes.

#    a) login_per_month : intensité d'usage normalisée par l'ancienneté.
#       Un client présent depuis 2 mois avec 20 logins est plus engagé
#       qu'un client présent depuis 24 mois avec 20 logins.
#       Le +1 au dénominateur évite la division par 0.
df_clean['login_per_month'] = df_clean['monthly_logins'] / (df_clean['tenure_months'] + 1)

#    b) payment_risk : impact financier des échecs de paiement.
#       Un échec à 100€ est plus problématique qu'un échec à 10€.
df_clean['payment_risk'] = df_clean['payment_failures'] * df_clean['monthly_fee']

#    c) recency_risk : combien de temps sans connexion vs durée habituelle d'une session.
#       Un client qui se connecte 30 min mais qui ne s'est pas connecté depuis 20 jours
#       est plus à risque qu'un client habituel à 1 min de session.
df_clean['recency_risk'] = df_clean['last_login_days_ago'] / (df_clean['avg_session_time'] + 1)

print("\nNouvelles features créées : login_per_month, payment_risk, recency_risk")
print(f"\nDimensions finales : {df_clean.shape}")
display(df_clean.head(3))


In [ ]:
# ============================================================
# CELLULE 8 — Encodage des variables catégorielles (One-Hot)
# ============================================================
# Les modèles scikit-learn et XGBoost ne savent pas manipuler des chaînes
# de caractères : il faut tout convertir en numérique.
# On utilise le One-Hot Encoding : une colonne binaire par modalité.

cat_cols = df_clean.select_dtypes(include=['object']).columns.tolist()
print(f"Variables catégorielles à encoder : {cat_cols}")

# pd.get_dummies fait le boulot proprement.
# drop_first=True : on supprime la 1ère modalité de chaque variable
# pour éviter la multicolinéarité (dummy variable trap).
# Exemple : pour "gender", on garde gender_Male et on déduit Female = (Male == 0).
df_encoded = pd.get_dummies(df_clean, columns=cat_cols, drop_first=True)

print(f"\nDimensions après encodage : {df_encoded.shape}")
print(f"Nouvelles colonnes créées : {df_encoded.shape[1] - df_clean.shape[1] + len(cat_cols)}")

# Vérification finale : il ne doit plus rester aucune colonne de type 'object'
print(f"Colonnes object restantes : {df_encoded.select_dtypes(include='object').shape[1]}")
display(df_encoded.head(3))


In [ ]:
# ============================================================
# CELLULE 9 — Split train/test + normalisation + sauvegarde
# ============================================================
# Dernière étape avant la modélisation : préparer les jeux d'entraînement
# et de test, normaliser les features, et tout sauvegarder pour le notebook
# suivant (02_models.ipynb) et pour le dashboard.

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import joblib
import os

# Séparation features / cible
X = df_encoded.drop(columns=['churn'])
y = df_encoded['churn']

print(f"Features (X) : {X.shape}")
print(f"Cible    (y) : {y.shape}")
print(f"Taux de churn : {y.mean()*100:.1f}%")

# Split stratifié — Très important quand le dataset est déséquilibré.
# stratify=y garantit qu'on aura ~10% de churn dans train ET dans test
# (sinon, par malchance, on pourrait se retrouver avec 5% / 15%).
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,        # 80% train / 20% test = standard
    random_state=42,      # reproductibilité (le 42 c'est du folklore data science)
    stratify=y            # préserve la proportion de churn
)

print(f"\nTrain : {X_train.shape[0]} lignes | churn : {y_train.mean()*100:.1f}%")
print(f"Test  : {X_test.shape[0]}  lignes | churn : {y_test.mean()*100:.1f}%")

# Normalisation : on met toutes les variables sur la même échelle.
# Indispensable pour la régression logistique et le MLP, neutre pour les arbres.
# RÈGLE D'OR : on .fit() UNIQUEMENT sur le train, jamais sur le test.
# Sinon le test "voit" la distribution de l'ensemble → data leakage → métriques optimistes.
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)   # apprend mean/std sur le train ET transforme
X_test_scaled  = scaler.transform(X_test)         # applique simplement les mean/std du train

# get_dummies a renvoyé des bool → on garde les noms de colonnes en repassant en DataFrame
X_train_scaled = pd.DataFrame(X_train_scaled, columns=X_train.columns)
X_test_scaled  = pd.DataFrame(X_test_scaled,  columns=X_test.columns)

print("\nNormalisation appliquée (StandardScaler)")
# Petit sanity check : après StandardScaler, mean ≈ 0 et std ≈ 1
print(f"Moyenne train (exemple tenure_months) : {X_train_scaled['tenure_months'].mean():.4f}")
print(f"Std    train  (exemple tenure_months) : {X_train_scaled['tenure_months'].std():.4f}")

# Sauvegarde de tout ce dont on aura besoin par la suite :
#  - les 4 fichiers X/y train/test pour 02_models.ipynb
#  - le scaler pour le dashboard (transformer un nouveau client en temps réel)
os.makedirs('../data/processed', exist_ok=True)
X_train_scaled.to_csv('../data/processed/X_train.csv', index=False)
X_test_scaled.to_csv('../data/processed/X_test.csv',   index=False)
y_train.to_csv('../data/processed/y_train.csv',        index=False)
y_test.to_csv('../data/processed/y_test.csv',          index=False)
joblib.dump(scaler, '../data/processed/scaler.pkl')

print("\nFichiers sauvegardés dans data/processed/ :")
print("  X_train.csv, X_test.csv, y_train.csv, y_test.csv, scaler.pkl")
